<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания  13



<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Inventory в C#, который будет представлять информацию о 
наличии товаров на складе. На основе этого класса разработать 2-3 производных 
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из 
классов  должны  быть  реализованы  новые  атрибуты  и  методы,  а  также 
переопределены  некоторые  методы  базового  класса  для  демонстрации 
полиморфизма.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

public class Item
{
    public int Id { get; }
    public string Name { get; }
    public double Volume { get; }

    public Item(int id, string name, double volume)
    {
        if (volume <= 0)
            throw new ArgumentException("Объем товара должен быть больше 0.", nameof(volume));

        Id = id;
        Name = name;
        Volume = volume;
    }

    public override string ToString()
    {
        return $"[{Id}] {Name} (Объем: {Volume} м³)";
    }
}

public class Inventory
{
    public int WarehouseId { get; set; }
    public string WarehouseName { get; set; }
    public double StorageCapacity { get; set; }

    protected List<Item> Items { get; } = new List<Item>();

    public double UsedCapacity => Items.Sum(i => i.Volume);

    public Inventory(int warehouseId, string warehouseName, double storageCapacity)
    {
        WarehouseId = warehouseId;
        WarehouseName = warehouseName;
        StorageCapacity = storageCapacity;
    }

    public virtual string GetStorageStatus()
    {
        double available = StorageCapacity - UsedCapacity;
        return $"Склад '{WarehouseName}' (ID: {WarehouseId}):\n" +
               $"  Занято: {UsedCapacity:F2} из {StorageCapacity:F2} м³ | " +
               $"Свободно: {available:F2} м³ (Товаров: {Items.Count} шт.)";
    }

    public virtual bool AddItem(Item item)
    {
        if (item == null)
        {
            Console.WriteLine("Ошибка: Попытка добавить пустой объект (null).");
            return false;
        }

        if (UsedCapacity + item.Volume > StorageCapacity)
        {
            Console.WriteLine($"[ОТКАЗ] Недостаточно места на складе '{WarehouseName}' для товара {item.Name}.");
            return false;
        }

        Items.Add(item);
        Console.WriteLine($"[УСПЕХ] Товар {item} добавлен на склад '{WarehouseName}'.");
        return true;
    }

    public virtual bool RemoveItem(Item item)
    {
        if (item == null) return false;

        var existingItem = Items.FirstOrDefault(i => i.Id == item.Id);
        if (existingItem != null)
        {
            Items.Remove(existingItem);
            Console.WriteLine($"[УДАЛЕНО] Товар {existingItem.Name} списан со склада '{WarehouseName}'.");
            return true;
        }

        Console.WriteLine($"[ОШИБКА] Товар {item.Name} не найден на складе '{WarehouseName}'.");
        return false;
    }
}

public class PersonalInventory : Inventory
{
    public string OwnerName { get; set; }

    public PersonalInventory(int warehouseId, string warehouseName, double storageCapacity, string ownerName)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        OwnerName = ownerName;
    }

    public override string GetStorageStatus()
    {
        string baseStatus = base.GetStorageStatus();
        return $"[ПЕРСОНАЛЬНЫЙ СКЛАД Владелец: {OwnerName}]\n{baseStatus}";
    }

    public void ChangeOwner(string newOwnerName)
    {
        Console.WriteLine($"[СМЕНА ВЛАДЕЛЬЦА] Владелец склада '{WarehouseName}' изменен с {OwnerName} на {newOwnerName}.");
        OwnerName = newOwnerName;
    }
}

public class GroupInventory : Inventory
{
    public string ProductGroup { get; set; }

    public GroupInventory(int warehouseId, string warehouseName, double storageCapacity, string productGroup)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        ProductGroup = productGroup;
    }

    public override bool AddItem(Item item)
    {
        Console.WriteLine($"--> Попытка размещения в секцию категории '{ProductGroup}'...");
        
        bool isAdded = base.AddItem(item);

        if (isAdded)
        {
            Console.WriteLine($"    [КАТЕГОРИЗАЦИЯ] Товар '{item.Name}' успешно привязан к группе '{ProductGroup}'.");
        }

        return isAdded;
    }

    public void PrintCategorySummary()
    {
        Console.WriteLine($"Групповой склад '{WarehouseName}' специализирован на категории: '{ProductGroup}'.");
    }
}

public class AutomatedInventory : Inventory
{
    public string AutomationLevel { get; set; }

    public AutomatedInventory(int warehouseId, string warehouseName, double storageCapacity, string automationLevel)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        AutomationLevel = automationLevel;
    }

    public override bool RemoveItem(Item item)
    {
        Console.WriteLine($"[РОБОТИЗИРОВАННАЯ ВЫГРУЗКА] Инициализация манипулятора (Уровень автоматизации: {AutomationLevel})...");

        bool isRemoved = base.RemoveItem(item);

        if (isRemoved)
        {
            Console.WriteLine($"    Автоматическая система уровня '{AutomationLevel}' завершила извлечение товара.");
        }

        return isRemoved;
    }

    public void RunDiagnostics()
    {
        Console.WriteLine($"[ДИАГНОСТИКА] Роботизированные системы склада '{WarehouseName}' (Уровень: {AutomationLevel}) функционируют нормально.");
    }
}

Console.WriteLine("        ДЕМОНСТРАЦИЯ РАБОТЫ ИЕРАРХИИ КЛАССОВ СКЛАДОВ\n");

Item laptop = new Item(101, "Ноутбук Dell", 0.5);
Item chair = new Item(102, "Офисное кресло", 4.0);
Item serverRack = new Item(103, "Серверная стойка", 15.0);

PersonalInventory personalWh = new PersonalInventory(1, "Склад-Бокс А1", 10.0, "Иван Петров");
GroupInventory groupWh = new GroupInventory(2, "Хаб Электроники", 50.0, "Компьютерная техника");
AutomatedInventory autoWh = new AutomatedInventory(3, "Робосклад-Север", 100.0, "Full AI Robotics");

Console.WriteLine("1. Полиморфная обработка через список базового типа List<Inventory>:\n");

List<Inventory> allWarehouses = new List<Inventory>
{
    personalWh,
    groupWh,
    autoWh
};

foreach (var wh in allWarehouses)
{
    Console.WriteLine($"Работаем со складом ID {wh.WarehouseId}");
    wh.AddItem(laptop);
    Console.WriteLine();
}

Console.WriteLine("Добавление остальных товаров:");
personalWh.AddItem(chair);
autoWh.AddItem(serverRack);
Console.WriteLine();

Console.WriteLine("2. Полиморфный опрос статуса хранилища (GetStorageStatus):\n");
foreach (var wh in allWarehouses)
{
    Console.WriteLine(wh.GetStorageStatus());
    Console.WriteLine(new string('-', 50));
}

Console.WriteLine("\n3. Полиморфное удаление товара:\n");
foreach (var wh in allWarehouses)
{
    wh.RemoveItem(laptop);
    Console.WriteLine();
}

Console.WriteLine("4. Вызов специализированных методов классов-наследников:\n");
personalWh.ChangeOwner("Алексей Смирнов");
groupWh.PrintCategorySummary();
autoWh.RunDiagnostics();

ДЕМОНСТРАЦИЯ РАБОТЫ ИЕРАРХИИ КЛАССОВ СКЛАДОВ

1. Полиморфная обработка через список базового типа List<Inventory>:

Работаем со складом ID 1
[УСПЕХ] Товар [101] Ноутбук Dell (Объем: 0,5 м³) добавлен на склад 'Склад-Бокс А1'.

Работаем со складом ID 2
--> Попытка размещения в секцию категории 'Компьютерная техника'...
[УСПЕХ] Товар [101] Ноутбук Dell (Объем: 0,5 м³) добавлен на склад 'Хаб Электроники'.
    [КАТЕГОРИЗАЦИЯ] Товар 'Ноутбук Dell' успешно привязан к группе 'Компьютерная техника'.

Работаем со складом ID 3
[УСПЕХ] Товар [101] Ноутбук Dell (Объем: 0,5 м³) добавлен на склад 'Робосклад-Север'.

Добавление остальных товаров:
[УСПЕХ] Товар [102] Офисное кресло (Объем: 4 м³) добавлен на склад 'Склад-Бокс А1'.
[УСПЕХ] Товар [103] Серверная стойка (Объем: 15 м³) добавлен на склад 'Робосклад-Север'.

2. Полиморфный опрос статуса хранилища (GetStorageStatus):

[ПЕРСОНАЛЬНЫЙ СКЛАД Владелец: Иван Петров]
Склад 'Склад-Бокс А1' (ID: 1):
  Занято: 4,50 из 10,00 м³ | Свободно: 5,50 м³ (Товаров: 2 шт.)
--------------------------------------------------
Склад 'Хаб Электроники' (ID: 2):
  Занято: 0,50 из 50,00 м³ | Свободно: 49,50 м³ (Товаров: 1 шт.)
--------------------------------------------------
Склад 'Робосклад-Север' (ID: 3):
  Занято: 15,50 из 100,00 м³ | Свободно: 84,50 м³ (Товаров: 2 шт.)
--------------------------------------------------

3. Полиморфное удаление товара:

[УДАЛЕНО] Товар Ноутбук Dell списан со склада 'Склад-Бокс А1'.

[УДАЛЕНО] Товар Ноутбук Dell списан со склада 'Хаб Электроники'.

[РОБОТИЗИРОВАННАЯ ВЫГРУЗКА] Инициализация манипулятора (Уровень автоматизации: Full AI Robotics)...
[УДАЛЕНО] Товар Ноутбук Dell списан со склада 'Робосклад-Север'.
    Автоматическая система уровня 'Full AI Robotics' завершила извлечение товара.

4. Вызов специализированных методов классов-наследников:

[СМЕНА ВЛАДЕЛЬЦА] Владелец склада 'Склад-Бокс А1' изменен с Иван Петров на Алексей Смирнов.
Групповой склад 'Хаб Электроники' специализирован на категории: 'Компьютерная техника'.
[ДИАГНОСТИКА] Роботизированные системы склада 'Робосклад-Север' (Уровень: Full AI Robotics) функционируют нормально.